# Submission figures and tables

This notebook reads completed Compact and Extended summaries and exports the figures and plotted data to one final-results directory. It does not run allocation algorithms.

The primary figure for each study is **mean error probability versus mean budget fraction**, shown separately for the one-dimensional and two-dimensional datasets.

In [ ]:
# Resolve the project whether Jupyter starts here or from the repository root.
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working = Path.cwd().resolve()
PROJECT = working / "problem-2-min-error-seq" if (working / "problem-2-min-error-seq").is_dir() else working
SOURCE = PROJECT / "src"
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))

from analysis import write_csv
from figures import load_summary, save_figures, summary_markdown

## Result selection

Use `test` to build a fully reproducible draft from the completed 180-cases-per-dimension runs. Change this value to `full` only after both full studies have completed successfully. Test and full exports are kept in separate directories.

In [ ]:
# Select the completed result scope used for every final table and figure.
RESULT_MODE = "test"
STUDIES = ("compact", "extended")
OUTPUT = PROJECT / "results" / "final" / RESULT_MODE

## Export matched summaries and figures

Each study retains its own matched sample. The exported CSV contains exactly the aggregate values used in its plots. Compact includes the Exhaustive reference and its error-gap figure; Extended contains the four scalable methods.

In [ ]:
# Load, validate, export, and display each study in a fixed order.
for study in STUDIES:
    summary = load_summary(PROJECT, study, mode=RESULT_MODE)
    if not summary:
        raise RuntimeError(f"The {study} {RESULT_MODE} summary is empty")

    dimensions = {int(row["dimension"]) for row in summary}
    if dimensions != {1, 2}:
        raise RuntimeError(f"The {study} summary must contain both spatial dimensions")

    study_output = OUTPUT / study
    write_csv(study_output / "tables" / "error_summary.csv", summary)

    display(Markdown(f"## {study.title()} results"))
    display(Markdown(summary_markdown(summary)))

    figures = save_figures(
        summary,
        study_output,
        include_exact=(study == "compact"),
    )
    for name, figure in figures.items():
        display(Markdown(f"### {name.replace('_', ' ').title()}"))
        display(figure)
        plt.close(figure)

print("Final exports:", OUTPUT)